In [ ]:
import searchconsole
import json
import pydata_google_auth
from pydata_google_auth import cache
from googleapiclient.discovery import build
from searchconsole.account import Account
import pandas as pd

with open('credentials_for_GSC.json') as f:
    cs = json.load(f)
    
credentials = pydata_google_auth.get_user_credentials(scopes=['https://www.googleapis.com/auth/webmasters.readonly'],
                                                     client_id=cs['installed']['client_id'],
                                                     client_secret=cs['installed']['client_secret'],
                                                     credentials_cache=cache.REAUTH,
                                                     auth_local_webserver=True)

service = build('searchconsole', 'v1', credentials=credentials)

account = Account(service, credentials)

### Список сайтів в консолі:

In [ ]:
account.webproperties

### Вибір сайту:

In [ ]:
webproperty = account['https://example.com/']

In [ ]:
# збираємо данні
query = webproperty.query.range(start='2024-05-01', stop='2024-06-30').dimension('page', 'query')
data = query.get()
len(data)

In [ ]:
#створюємо data frame з тим що отримати
df = data.to_dataframe()
df.head()
#df.to_excel('raw_data.xlsx') #якщо вже потрібно вигрузити данні то прибираємо "#"

In [ ]:
#Фільтруємо дані по списку стоп-слів
#patternDel = ".*brand.*"
#filter = df['query'].str.contains(patternDel)
#filtered_df = df[~filter]

#Видаляємо не потрібні сторінки
#patternDel = ".*/spiders/.*"
#filtered_df_from_pages = filtered_df['page'].str.contains(patternDel) видлаеня по списку сторінок й по списку стоп слів
#filtered_df_from_pages = df['page'].str.contains(patternDel) #видаленя просто по списку сторінок без видаленя по списку стоп слів
#df = filtered_df_from_pages

df_filter_impressions = df[df['impressions'] >= 5] #беремо покази від 10
df_filter_clicks = df_filter_impressions[df_filter_impressions['clicks'] >= 0] #беремо кліки від 1
df_filter_clicks.head()

In [ ]:
#Рахуємо кільсть сторінок, запитів, кліків, показів, середній CTR и середню позицію
data = {
    'Total pages': [int(df_filter_clicks['page'].nunique())],
    'Total queries': [int(df_filter_clicks['query'].nunique())],
    'Total clicks': [int(df_filter_clicks['clicks'].sum())],
    'Total impressions': [int(df_filter_clicks['impressions'].sum())],
    'Average CTR': [round(df_filter_clicks['ctr'].mean(),2)],
    'Average position': [round(df_filter_clicks['position'].mean(),2)],
    'Average queries per page': [round(int(df_filter_clicks['query'].nunique()) / int(df_filter_clicks['page'].nunique()),2)],
}

df_filter_clicks_stats = pd.DataFrame.from_dict(data)
df_filter_clicks_stats.T.head(10)

In [ ]:
#рахуєм суму унікальних сторінок по кажному ключу, також рахуємо суму кликів, показів, середній CTR й позицію
df_summary = df_filter_clicks.groupby('query').agg(
    unique_pages=('page', 'nunique'),
    total_clicks=('clicks', 'sum'),
    total_impressions=('impressions', 'sum'),
    avg_ctr=('ctr', 'mean'),
    avg_position=('position', 'mean'),    
).sort_values(by='total_clicks', ascending=False)

df_summary.head()

In [ ]:
#отримаємо запити де канібалізація
df_cannibalised = df_filter_clicks.groupby('query').agg(
    page = ('page', 'unique'),
    unique_pages=('page', 'nunique'),
    total_clicks=('clicks', 'sum'),
    total_impressions=('impressions', 'sum'),
    avg_ctr=('ctr', 'mean'),
    avg_position=('position', 'mean'),    
).sort_values(by='unique_pages', ascending=False)

df_cannibalised = df_cannibalised[df_cannibalised['unique_pages'] > 1] #берем страницы где значение unique_pages больше 1
df_cannibalised = df_cannibalised[df_cannibalised['unique_pages'] < 10] #берем страницы где значение не больше 10 (чтобы не смотреть все страницы, можно эту строку закомментить)
df_cannibalised.head()

In [ ]:
final = df_cannibalised.explode('page')
final.drop(['avg_ctr', 'avg_position', 'total_impressions', 'total_clicks', 'unique_pages'], axis=1, inplace=True)
final.head()

In [ ]:
#Створюємо красиву таблицю
data_zapis = pd.merge(final, df, on=['query', 'page'],how = "left")
data_zapis.head()

In [ ]:
new_data = data_zapis.set_index(['query', 'page', 'clicks', 'impressions', 'ctr', 'position'])
new_data.head()

In [ ]:
#записуємо результат
new_data.to_excel('cannibalization.xlsx')